# Data Quality and Exploratory Analysis: Customer Churn

**Objective:** produce a reproducible, analysis-ready dataset and a defensible data-quality report.

**Inputs** (untouched, in `data_raw/`): `customer_churn_dataset-training-master.csv` (primary) and `customer_churn_dataset-testing-master.csv`.
**Outputs:** `data_clean/churn_train_clean.csv`, `data_clean/churn_test_clean.csv`, `charts/*.png`, `data_quality_summary.md`.

Run top to bottom to reproduce everything.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style='whitegrid')
RAW = Path('data_raw'); CLEAN = Path('data_clean'); CH = Path('charts')
CLEAN.mkdir(exist_ok=True); CH.mkdir(exist_ok=True)

NUM = ['Age','Tenure','Usage Frequency','Support Calls','Payment Delay','Total Spend','Last Interaction']
CAT = ['Gender','Subscription Type','Contract Length']

train_raw = pd.read_csv(RAW/'customer_churn_dataset-training-master.csv')
test_raw  = pd.read_csv(RAW/'customer_churn_dataset-testing-master.csv')
print('train', train_raw.shape, '| test', test_raw.shape)

train (440833, 12) | test (64374, 12)


## 1. Profile the data

For each file: types, missing values, duplicates, ranges, categories and outliers.

In [ ]:
def profile(df, name):
    print(f'===== {name} =====')
    print('shape:', df.shape)
    print('fully blank rows:', int(df.isna().all(axis=1).sum()),
          '| rows with any missing:', int(df.isna().any(axis=1).sum()))
    print('exact duplicate rows:', int(df.duplicated().sum()),
          '| duplicate CustomerID:', int(df['CustomerID'].duplicated().sum()),
          '| duplicates ignoring ID:', int(df.drop(columns='CustomerID').duplicated().sum()))
    print('whitespace/case issues in categoricals:',
          int(sum((df[c].dropna()!=df[c].dropna().str.strip()).sum() for c in CAT)))
    out = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(),
                        'unique': df.nunique()})
    display(out)
    display(df[NUM+['Churn']].describe().T.round(2))
    for c in CAT+['Churn']:
        print(c, df[c].value_counts(dropna=False).to_dict())

profile(train_raw, 'TRAIN (raw)')
profile(test_raw, 'TEST (raw)')

===== TRAIN (raw) =====
shape: (440833, 12)
fully blank rows: 1 | rows with any missing: 1


exact duplicate rows: 0 | duplicate CustomerID: 0 | duplicates ignoring ID: 0


whitespace/case issues in categoricals: 0


,dtype,missing,unique
CustomerID,float64,1,440832
Age,float64,1,48
Gender,str,1,2
Tenure,float64,1,60
Usage Frequency,float64,1,30
Support Calls,float64,1,11
Payment Delay,float64,1,31
Subscription Type,str,1,3
Contract Length,str,1,3
Total Spend,float64,1,68363


,count,mean,std,min,25%,50%,75%,max
Age,440832.0,39.37,12.44,18.0,29.0,39.0,48.0,65.0
Tenure,440832.0,31.26,17.26,1.0,16.0,32.0,46.0,60.0
Usage Frequency,440832.0,15.81,8.59,1.0,9.0,16.0,23.0,30.0
Support Calls,440832.0,3.60,3.07,0.0,1.0,3.0,6.0,10.0
Payment Delay,440832.0,12.97,8.26,0.0,6.0,12.0,19.0,30.0
Total Spend,440832.0,631.62,240.80,100.0,480.0,661.0,830.0,1000.0
Last Interaction,440832.0,14.48,8.60,1.0,7.0,14.0,22.0,30.0
Churn,440832.0,0.57,0.50,0.0,0.0,1.0,1.0,1.0


Gender {'Male': 250252, 'Female': 190580, nan: 1}
Subscription Type {'Standard': 149128, 'Premium': 148678, 'Basic': 143026, nan: 1}
Contract Length {'Annual': 177198, 'Quarterly': 176530, 'Monthly': 87104, nan: 1}
Churn {1.0: 249999, 0.0: 190833, nan: 1}
===== TEST (raw) =====
shape: (64374, 12)
fully blank rows: 0 | rows with any missing: 0
exact duplicate rows: 0 | duplicate CustomerID: 0 | duplicates ignoring ID: 0
whitespace/case issues in categoricals: 0


,dtype,missing,unique
CustomerID,int64,0,64374
Age,int64,0,48
Gender,str,0,2
Tenure,int64,0,60
Usage Frequency,int64,0,30
Support Calls,int64,0,11
Payment Delay,int64,0,31
Subscription Type,str,0,3
Contract Length,str,0,3
Total Spend,int64,0,901


,count,mean,std,min,25%,50%,75%,max
Age,64374.0,41.97,13.92,18.0,30.0,42.0,54.0,65.0
Tenure,64374.0,31.99,17.10,1.0,18.0,33.0,47.0,60.0
Usage Frequency,64374.0,15.08,8.82,1.0,7.0,15.0,23.0,30.0
Support Calls,64374.0,5.40,3.11,0.0,3.0,6.0,8.0,10.0
Payment Delay,64374.0,17.13,8.85,0.0,10.0,19.0,25.0,30.0
Total Spend,64374.0,541.02,260.87,100.0,313.0,534.0,768.0,1000.0
Last Interaction,64374.0,15.50,8.64,1.0,8.0,15.0,23.0,30.0
Churn,64374.0,0.47,0.50,0.0,0.0,0.0,1.0,1.0


Gender {'Female': 34353, 'Male': 30021}
Subscription Type {'Standard': 21502, 'Basic': 21451, 'Premium': 21421}
Contract Length {'Monthly': 22130, 'Annual': 21410, 'Quarterly': 20834}
Churn {0: 33881, 1: 30493}


In [ ]:
# Outliers: Tukey IQR rule (1.5 x IQR) on every numeric column, plus range checks
rows = []
for c in NUM:
    s = train_raw[c].dropna()
    q1, q3 = s.quantile([.25,.75]); iqr = q3-q1
    lo, hi = q1-1.5*iqr, q3+1.5*iqr
    rows.append({'column': c, 'min': s.min(), 'max': s.max(), 'IQR_lower': lo, 'IQR_upper': hi,
                 'n_outliers': int(((s<lo)|(s>hi)).sum()), 'skew': round(s.skew(),2)})
display(pd.DataFrame(rows))

# Keys across the two files
tr_ids = set(train_raw['CustomerID'].dropna().astype(int)); te_ids = set(test_raw['CustomerID'])
overlap = tr_ids & te_ids
m = train_raw.dropna().astype({'CustomerID':int}).merge(test_raw, on='CustomerID', suffixes=('_tr','_te'))
same = ((m.Age_tr==m.Age_te)&(m.Gender_tr==m.Gender_te)&(m.Tenure_tr==m.Tenure_te)).mean()
print(f'CustomerIDs appearing in both files: {len(overlap):,}')
print(f'...of which describe the same customer (age, gender, tenure match): {same:.4%}')

,column,min,max,IQR_lower,IQR_upper,n_outliers,skew
0,Age,18.0,65.0,0.5,76.5,0,0.16
1,Tenure,1.0,60.0,-29.0,91.0,0,-0.06
2,Usage Frequency,1.0,30.0,-12.0,44.0,0,-0.04
3,Support Calls,0.0,10.0,-6.5,13.5,0,0.67
4,Payment Delay,0.0,30.0,-13.5,38.5,0,0.27
5,Total Spend,100.0,1000.0,-45.0,1355.0,0,-0.46
6,Last Interaction,1.0,30.0,-15.5,44.5,0,0.18


CustomerIDs appearing in both files: 62,995
...of which describe the same customer (age, gender, tenure match): 0.0143%


**Profiling findings**

| # | Finding | Evidence |
|---|---|---|
| 1 | One completely blank row in the training file | 1 row, all 12 columns null |
| 2 | Training columns load as `float64` only because of that blank row | all integer-valued except `Total Spend` |
| 3 | `CustomerID` is **not** a shared key across files | ~63k IDs appear in both files but belong to different people |
| 4 | No duplicates, no whitespace/case problems, no IQR outliers | all columns bounded (e.g. Age 18-65, Tenure 1-60) |
| 5 | Train and test differ in distribution | see section 3 |

## 2. Clean the data (documented assumptions + validation)

In [ ]:
assumptions = []
def log(step, issue, action, rationale, before, after):
    assumptions.append(dict(step=step, issue=issue, action=action, rationale=rationale,
                            rows_before=before, rows_after=after))

def clean(df, source):
    d = df.copy()
    n0 = len(d)

    # A1. Drop fully blank rows (no information, cannot be imputed)
    d = d.dropna(how='all')
    log(f'{source}-A1', 'Fully blank row(s)', 'Dropped', 'A row with every field null carries no information; imputing would invent a customer', n0, len(d))

    # A2. Enforce integer types on count-like fields (floats only came from the blank row)
    n1 = len(d)
    int_cols = ['CustomerID','Age','Tenure','Usage Frequency','Support Calls','Payment Delay','Last Interaction','Churn']
    d[int_cols] = d[int_cols].astype('int64')
    log(f'{source}-A2', 'Integer fields stored as float', 'Cast to int64', 'Values were all whole numbers; floats existed only because of NaN', n1, len(d))

    # A3. Standardise categoricals (strip, Title Case) and use category dtype
    n2 = len(d)
    for c in CAT:
        d[c] = d[c].str.strip().str.title().astype('category')
    log(f'{source}-A3', 'Categorical text hygiene', 'Strip + Title Case + category dtype', 'No variants found, applied defensively so the pipeline is reusable', n2, len(d))

    # A4. Deduplicate (exact rows and repeated IDs within a file)
    n3 = len(d)
    d = d.drop_duplicates().drop_duplicates(subset='CustomerID', keep='first')
    log(f'{source}-A4', 'Duplicate rows / repeated IDs inside file', 'drop_duplicates (none found)', 'Safety net; zero rows removed on this data', n3, len(d))

    # A5. Namespaced key because IDs collide across files
    d.insert(0, 'customer_key', f'{source}_' + d['CustomerID'].astype(str))
    d['source'] = source
    log(f'{source}-A5', 'CustomerID collides across train/test', 'Added customer_key = <source>_<id> and a source column', 'The same numeric ID refers to different people in each file', len(d), len(d))

    # A6. Outliers: retained
    log(f'{source}-A6', 'Outliers', 'None removed', 'Zero IQR outliers; all values within plausible business ranges', len(d), len(d))

    return d.reset_index(drop=True)

train = clean(train_raw, 'train')
test  = clean(test_raw,  'test')
display(pd.DataFrame(assumptions)[['step','issue','action','rows_before','rows_after']])

,step,issue,action,rows_before,rows_after
0,train-A1,Fully blank row(s),Dropped,440833,440832
1,train-A2,Integer fields stored as float,Cast to int64,440832,440832
2,train-A3,Categorical text hygiene,Strip + Title Case + category dtype,440832,440832
3,train-A4,Duplicate rows / repeated IDs inside file,drop_duplicates (none found),440832,440832
4,train-A5,CustomerID collides across train/test,Added customer_key = <source>_<id> and a sourc...,440832,440832
5,train-A6,Outliers,None removed,440832,440832
6,test-A1,Fully blank row(s),Dropped,64374,64374
7,test-A2,Integer fields stored as float,Cast to int64,64374,64374
8,test-A3,Categorical text hygiene,Strip + Title Case + category dtype,64374,64374
9,test-A4,Duplicate rows / repeated IDs inside file,drop_duplicates (none found),64374,64374


In [ ]:
# Validation checks: the script fails loudly if any rule is broken
def validate(d, name):
    checks = {
        'no missing values':            d.isna().sum().sum() == 0,
        'customer_key unique':          d['customer_key'].is_unique,
        'Age in [18, 65]':              d['Age'].between(18, 65).all(),
        'Tenure in [1, 60]':            d['Tenure'].between(1, 60).all(),
        'Usage Frequency in [1, 30]':   d['Usage Frequency'].between(1, 30).all(),
        'Support Calls in [0, 10]':     d['Support Calls'].between(0, 10).all(),
        'Payment Delay in [0, 30]':     d['Payment Delay'].between(0, 30).all(),
        'Last Interaction in [1, 30]':  d['Last Interaction'].between(1, 30).all(),
        'Total Spend in [100, 1000]':   d['Total Spend'].between(100, 1000).all(),
        'Churn is 0/1':                 d['Churn'].isin([0, 1]).all(),
        'Gender values valid':          set(d['Gender'].unique()) <= {'Male','Female'},
        'Subscription values valid':    set(d['Subscription Type'].unique()) <= {'Basic','Standard','Premium'},
        'Contract values valid':        set(d['Contract Length'].unique()) <= {'Monthly','Quarterly','Annual'},
    }
    res = pd.Series(checks, name=name)
    assert res.all(), f'Validation failed for {name}:\n{res[~res]}'
    return res

val = pd.concat([validate(train,'train'), validate(test,'test')], axis=1)
display(val.replace({True:'PASS'}))
print('Row reconciliation  train:', len(train_raw), '->', len(train), '| test:', len(test_raw), '->', len(test))

,train,test
no missing values,PASS,PASS
customer_key unique,PASS,PASS
"Age in [18, 65]",PASS,PASS
"Tenure in [1, 60]",PASS,PASS
"Usage Frequency in [1, 30]",PASS,PASS
"Support Calls in [0, 10]",PASS,PASS
"Payment Delay in [0, 30]",PASS,PASS
"Last Interaction in [1, 30]",PASS,PASS
"Total Spend in [100, 1000]",PASS,PASS
Churn is 0/1,PASS,PASS


Row reconciliation  train: 440833 -> 440832 | test: 64374 -> 64374


In [ ]:
train.to_csv(CLEAN/'churn_train_clean.csv', index=False)
test.to_csv(CLEAN/'churn_test_clean.csv', index=False)
pd.DataFrame(assumptions).to_csv(CLEAN/'cleaning_log.csv', index=False)
print('saved:', sorted(p.name for p in CLEAN.iterdir()))

saved: ['churn_test_clean.csv', 'churn_train_clean.csv', 'cleaning_log.csv']


## 3. Summary statistics and distributions

The analysis below uses the **training file** (440,832 customers after cleaning). The test file is compared separately because its population differs.

In [ ]:
overall = train['Churn'].mean()
print(f'Overall churn rate (train): {overall:.1%}  ({train.Churn.sum():,} of {len(train):,})')
print(f'Overall churn rate (test):  {test.Churn.mean():.1%}')
display(train[NUM].describe().T.round(2))
display(train.groupby('Churn')[NUM].mean().T.round(1).rename(columns={0:'Retained',1:'Churned'}))
corr = train[NUM+['Churn']].corr()['Churn'].drop('Churn').sort_values(key=abs, ascending=False).round(3)
print('Correlation with Churn:'); display(corr.to_frame('r'))

Overall churn rate (train): 56.7%  (249,999 of 440,832)
Overall churn rate (test):  47.4%


,count,mean,std,min,25%,50%,75%,max
Age,440832.0,39.37,12.44,18.0,29.0,39.0,48.0,65.0
Tenure,440832.0,31.26,17.26,1.0,16.0,32.0,46.0,60.0
Usage Frequency,440832.0,15.81,8.59,1.0,9.0,16.0,23.0,30.0
Support Calls,440832.0,3.60,3.07,0.0,1.0,3.0,6.0,10.0
Payment Delay,440832.0,12.97,8.26,0.0,6.0,12.0,19.0,30.0
Total Spend,440832.0,631.62,240.80,100.0,480.0,661.0,830.0,1000.0
Last Interaction,440832.0,14.48,8.60,1.0,7.0,14.0,22.0,30.0


Churn,Retained,Churned
Age,36.3,41.7
Tenure,32.3,30.5
Usage Frequency,16.3,15.5
Support Calls,1.6,5.1
Payment Delay,10.0,15.2
Total Spend,750.0,541.3
Last Interaction,13.0,15.6


Correlation with Churn:


,r
Support Calls,0.574
Total Spend,-0.429
Payment Delay,0.312
Age,0.218
Last Interaction,0.150
Tenure,-0.052
Usage Frequency,-0.046


In [ ]:
# Churn rate by segment
def seg(col, bins=None, labels=None):
    s = pd.cut(train[col], bins=bins, labels=labels) if bins is not None else train[col]
    out = train.groupby(s, observed=True)['Churn'].agg(rate='mean', customers='size')
    out['rate'] = (out['rate']*100).round(1)
    return out

for col in CAT: print(col); display(seg(col))
print('Support Calls'); display(seg('Support Calls', [-1,2,5,10], ['0-2','3-5','6-10']))
print('Age'); display(seg('Age', [17,25,35,45,55,65], ['18-25','26-35','36-45','46-55','56-65']))
print('Payment Delay (days)'); display(seg('Payment Delay', [-1,10,20,30], ['0-10','11-20','21-30']))
print('Tenure (months)'); display(seg('Tenure', [0,12,24,36,48,60], ['1-12','13-24','25-36','37-48','49-60']))

Gender


,rate,customers
Gender,,
Female,66.7,190580
Male,49.1,250252


Subscription Type


,rate,customers
Subscription Type,,
Basic,58.2,143026
Premium,55.9,148678
Standard,56.1,149128


Contract Length


...

,rate,customers
Age,,
18-25,56.4,73683
26-35,49.1,104108
36-45,42.8,118413
46-55,57.8,90203
56-65,100.0,54425


Payment Delay (days)


,rate,customers
Payment Delay,,
0-10,46.5,186247
11-20,46.5,170555
21-30,100.0,84030


Tenure (months)


,rate,customers
Tenure,,
1-12,58.7,85129
13-24,63.3,79967
25-36,54.1,91473
37-48,54.1,91932
49-60,54.3,92331


In [ ]:
# Chart 1: numeric distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8)); axes = axes.ravel()
for ax, c in zip(axes, NUM):
    sns.histplot(train[c], bins=30, ax=ax, color='#4C72B0'); ax.set_title(c)
axes[-1].axis('off')
fig.suptitle('Distributions of numeric fields (training, n=440,832)', fontsize=14, y=1.0)
plt.tight_layout(); plt.savefig(CH/'01_numeric_distributions.png', dpi=130); plt.close()

# Chart 2: boxplots by churn
fig, axes = plt.subplots(2, 4, figsize=(18, 8)); axes = axes.ravel()
for ax, c in zip(axes, NUM):
    sns.boxplot(data=train, x='Churn', y=c, ax=ax, hue='Churn', palette=['#55A868','#C44E52'], legend=False)
    ax.set_xticks([0,1], ['Retained','Churned']); ax.set_xlabel(''); ax.set_title(c)
axes[-1].axis('off')
fig.suptitle('Numeric fields by churn outcome', fontsize=14, y=1.0)
plt.tight_layout(); plt.savefig(CH/'02_boxplots_by_churn.png', dpi=130); plt.close()

# Chart 3: churn rate by categorical fields
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, c in zip(axes, CAT):
    r = train.groupby(c, observed=True)['Churn'].mean().mul(100)
    sns.barplot(x=r.index.astype(str), y=r.values, ax=ax, color='#4C72B0')
    ax.axhline(overall*100, ls='--', c='grey'); ax.set_ylim(0,110); ax.set_ylabel('Churn rate (%)'); ax.set_xlabel(''); ax.set_title(c)
    for i, v in enumerate(r.values): ax.text(i, v+1.5, f'{v:.0f}%', ha='center')
fig.suptitle('Churn rate by category (dashed = overall)', fontsize=13)
plt.tight_layout(); plt.savefig(CH/'03_churn_by_category.png', dpi=130); plt.close()

# Chart 4: churn rate by key numeric bands
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
specs = [('Support Calls', [-1,2,5,10], ['0-2','3-5','6-10']),
         ('Age', [17,25,35,45,55,65], ['18-25','26-35','36-45','46-55','56-65']),
         ('Payment Delay', [-1,10,20,30], ['0-10','11-20','21-30'])]
for ax, (c, b, l) in zip(axes, specs):
    r = train.groupby(pd.cut(train[c], b, labels=l), observed=True)['Churn'].mean().mul(100)
    sns.barplot(x=r.index.astype(str), y=r.values, ax=ax, color='#C44E52')
    ax.axhline(overall*100, ls='--', c='grey'); ax.set_ylim(0,110); ax.set_title(c); ax.set_xlabel(''); ax.set_ylabel('Churn rate (%)')
    for i, v in enumerate(r.values): ax.text(i, v+1.5, f'{v:.0f}%', ha='center')
fig.suptitle('Churn rate by band', fontsize=13)
plt.tight_layout(); plt.savefig(CH/'04_churn_by_bands.png', dpi=130); plt.close()

# Chart 5: correlation heatmap
plt.figure(figsize=(8, 6.5))
sns.heatmap(train[NUM+['Churn']].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation matrix (training)'); plt.tight_layout(); plt.savefig(CH/'05_correlation_heatmap.png', dpi=130); plt.close()

# Chart 6: train vs test shift
shift_cols = ['Support Calls','Payment Delay','Total Spend','Age']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, c in zip(axes, shift_cols):
    sns.kdeplot(train[c], ax=ax, label='train', fill=True, alpha=.3)
    sns.kdeplot(test[c],  ax=ax, label='test',  fill=True, alpha=.3); ax.set_title(c); ax.legend()
fig.suptitle('Train vs test distribution shift', fontsize=13)
plt.tight_layout(); plt.savefig(CH/'06_train_vs_test_shift.png', dpi=130); plt.close()
print('charts saved:', sorted(p.name for p in CH.iterdir()))

charts saved: ['01_numeric_distributions.png', '02_boxplots_by_churn.png', '03_churn_by_category.png', '04_churn_by_bands.png', '05_correlation_heatmap.png', '06_train_vs_test_shift.png']


In [ ]:
# Perfect-churn thresholds (a major insight, and a modelling risk)
rules = {
  'Contract Length == Monthly': train['Contract Length']=='Monthly',
  'Support Calls >= 6':         train['Support Calls']>=6,
  'Age >= 56':                  train['Age']>=56,
  'Payment Delay >= 21':        train['Payment Delay']>=21,
}
rows = [{'rule': k, 'customers': int(v.sum()), 'share_of_base_%': round(v.mean()*100,1),
         'churn_rate_%': round(train.loc[v,'Churn'].mean()*100,1)} for k, v in rules.items()]
any_rule = np.logical_or.reduce(list(rules.values()))
rows.append({'rule': 'ANY of the four rules', 'customers': int(any_rule.sum()),
             'share_of_base_%': round(any_rule.mean()*100,1), 'churn_rate_%': round(train.loc[any_rule,'Churn'].mean()*100,1)})
rows.append({'rule': 'NONE of the four rules', 'customers': int((~any_rule).sum()),
             'share_of_base_%': round((~any_rule).mean()*100,1), 'churn_rate_%': round(train.loc[~any_rule,'Churn'].mean()*100,1)})
display(pd.DataFrame(rows))

# Train vs test comparison
cmp = pd.DataFrame({'train': train[NUM+['Churn']].mean(), 'test': test[NUM+['Churn']].mean()}).round(2)
cmp['diff_%'] = ((cmp['test']/cmp['train']-1)*100).round(1)
display(cmp)

,rule,customers,share_of_base_%,churn_rate_%
0,Contract Length == Monthly,87104,19.8,100.0
1,Support Calls >= 6,118513,26.9,100.0
2,Age >= 56,54425,12.3,100.0
3,Payment Delay >= 21,84030,19.1,100.0
4,ANY of the four rules,209820,47.6,100.0
5,NONE of the four rules,231012,52.4,17.4


,train,test,diff_%
Age,39.37,41.97,6.6
Tenure,31.26,31.99,2.3
Usage Frequency,15.81,15.08,-4.6
Support Calls,3.60,5.40,50.0
Payment Delay,12.97,17.13,32.1
Total Spend,631.62,541.02,-14.3
Last Interaction,14.48,15.50,7.0
Churn,0.57,0.47,-17.5
